# Run Log Manager
Tracks all QRUMBLE random search runs across rounds and seeds.

## 1. Setup

In [ ]:
import os
import glob
import pandas as pd

# ── Preset names ──────────────────────────────────────────────────────────────
# Edit these if you add new presets
eq   = 'eq'    # uniform              [0.34, 0.33, 0.33]
prob_1  = 'dm1'   # strongly negative    [0.10, 0.10, 0.80]
prob1  = 'dp1'   # strongly positive    [0.10, 0.80, 0.10]
prob0  = 'dm0'  # not relevant      [0.80, 0.10, 0.10]

# ── Path ──────────────────────────────────────────────────────────────────────
LOG_PATH = 'logs/run_log.csv'
COLUMNS  = ['round', 'seed', 'n_evals', 'cache_total',
             'pfactors', 'best_sharpe', 'best_return', 'best_var', 'notes']

os.makedirs('logs', exist_ok=True)

if not os.path.exists(LOG_PATH):
    pd.DataFrame(columns=COLUMNS).to_csv(LOG_PATH, index=False)
    print('✅ Created new run_log.csv')
else:
    print(f'✅ Loaded existing run_log.csv  ({sum(1 for _ in open(LOG_PATH)) - 1} rows)')

# ── Helper functions ──────────────────────────────────────────────────────────

def _load():
    df = pd.read_csv(LOG_PATH, dtype={'round': int, 'seed': int, 'n_evals': int,
                                       'cache_total': int})
    return df

def _save(df):
    df.to_csv(LOG_PATH, index=False)

def _pfactors_str(pf_list):
    """Convert a list of preset names to the canonical string."""
    inner = ', '.join(pf_list)
    return f'pfactors = [{inner}]'

def _recompute_cache(df):
    """Recompute cache_total as cumulative sum of n_evals in CSV order."""
    df = df.copy()
    df['cache_total'] = df['n_evals'].cumsum()
    return df

def add_row(round, seed, n_evals, pfactors,
            best_sharpe=None, best_return=None, best_var=None, notes=''):
    """Append a new row. cache_total is computed automatically."""
    df = _load()
    pf_str = _pfactors_str(pfactors)
    cache  = int(df['n_evals'].sum()) + n_evals if len(df) else n_evals
    new_row = pd.DataFrame([{
        'round': round, 'seed': seed, 'n_evals': n_evals,
        'cache_total': cache, 'pfactors': pf_str,
        'best_sharpe': best_sharpe, 'best_return': best_return,
        'best_var': best_var, 'notes': notes
    }])
    df = pd.concat([df, new_row], ignore_index=True)
    _save(df)
    print(f'✅ Row added  (round={round}, seed={seed})  cache_total = {cache}')

def edit_row(round, seed, **kwargs):
    """Edit any field(s) of the row identified by (round, seed)."""
    df = _load()
    mask = (df['round'] == round) & (df['seed'] == seed)
    if not mask.any():
        print(f'❌ No row found for round={round}, seed={seed}')
        return
    for col, val in kwargs.items():
        if col == 'pfactors' and isinstance(val, list):
            val = _pfactors_str(val)
        if col not in df.columns:
            print(f'⚠️  Unknown column "{col}" — skipped')
            continue
        df.loc[mask, col] = val
    _save(df)
    print(f'✅ Row updated  (round={round}, seed={seed})  fields: {list(kwargs.keys())}')

def delete_row(round, seed):
    """Delete the row for (round, seed) and recompute cache_total."""
    df = _load()
    mask = (df['round'] == round) & (df['seed'] == seed)
    if not mask.any():
        print(f'❌ No row found for round={round}, seed={seed}')
        return
    df = df[~mask].reset_index(drop=True)
    df = _recompute_cache(df)
    _save(df)
    print(f'✅ Row deleted  (round={round}, seed={seed})  {len(df)} rows remaining')

def fill_metrics_from_csv(round, seed, csv_path):
    """Read a QRUMBLE raw CSV and fill best_sharpe, best_return, best_var."""
    if not os.path.exists(csv_path):
        print(f'❌ File not found: {csv_path}')
        return
    raw = pd.read_csv(csv_path)
    for col in ('sharpe', 'annualized', 'var'):
        if col not in raw.columns:
            print(f'❌ Column "{col}" not found in {csv_path}')
            return
    best_idx    = raw['sharpe'].idxmax()
    best_sharpe = round(float(raw.loc[best_idx, 'sharpe']),    4)
    best_return = round(float(raw.loc[best_idx, 'annualized']),4)
    best_var    = round(float(raw.loc[best_idx, 'var']),       4)
    edit_row(round, seed,
             best_sharpe=best_sharpe,
             best_return=best_return,
             best_var=best_var)
    print(f'   sharpe={best_sharpe}  return={best_return}  var={best_var}')


✅ Loaded existing run_log.csv  (6 rows)


## 2. View Log

In [38]:
df = _load()
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)
display(df)
print(f'\n{len(df)} rows  |  Total evaluations: {df["n_evals"].sum() if len(df) else 0}')


,round,seed,n_evals,cache_total,pfactors,best_sharpe,best_return,best_var,notes
0,1,42,2000,1991,"pfactors = [eq, eq, eq, eq, eq, eq, eq, eq, eq, eq, eq]",1.46,29.76,-27.62,uniform round 1
1,1,123,2000,3982,"pfactors = [eq, eq, eq, eq, eq, eq, eq, eq, eq, eq, eq]",1.46,29.76,-27.62,uniform round 1
2,1,999,2000,5973,unknown,1.46,29.76,-27.62,NaN
3,2,43,500,6460,unknown,1.30,26.81,-29.39,NaN
4,2,124,500,6947,unknown,1.47,32.47,-30.20,NaN
5,2,998,500,7434,unknown,1.47,32.47,-30.20,NaN



6 rows  |  Total evaluations: 7500


## 3. Add a Row
Fill in the values below and run the cell.

In [ ]:
add_row(
    round       = 1,
    seed        = 42,
    n_evals     = 2000,
    pfactors    = [eq, eq, eq, eq, eq, eq, eq, eq, eq, eq, eq],
    best_sharpe = None,
    best_return = None,
    best_var    = None,
    notes       = 'uniform round 1'
)


✅ Row added  (round=1, seed=123)  cache_total = 3991


/var/folders/r3/6w1hfmjj2jdc4xp0rcj6p8fr0000gn/T/ipykernel_85908/2851863482.py:60: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, new_row], ignore_index=True)


## 4. Edit a Row
Pass any fields you want to change as keyword arguments. Others are left as-is.

In [ ]:
edit_row(
    round       = 2,
    seed        = 998,
    pfactors =  [prob_1, prob_1, eq, eq, prob_1, eq, eq, prob0, eq, eq, prob_1]
)


✅ Row updated  (round=1, seed=999)  fields: ['pfactors']


## 5. Delete a Row

In [ ]:
delete_row(
    round = 1,
    seed  = 42
)


TypeError: delete_row() got an unexpected keyword argument 'n_evals'

## 6. Auto-fill Metrics from Raw CSV
Reads a QRUMBLE output CSV and fills best_sharpe, best_return, best_var for a given (round, seed).

In [24]:
fill_metrics_from_csv(
    round    = 1,
    seed     = 42,
    csv_path = '../search/outputs/som/round1_seed123_2000_raw.csv'
)


TypeError: 'int' object is not callable

## 7. Summary Table
Group by round — total evals, mean/max Sharpe, cumulative cache at end of each round.

In [25]:
df = _load()

if len(df) == 0:
    print('No data yet.')
else:
    summary = (
        df.groupby('round')
        .agg(
            seeds        = ('seed',         'count'),
            total_evals  = ('n_evals',      'sum'),
            cache_end    = ('cache_total',  'max'),
            sharpe_mean  = ('best_sharpe',  'mean'),
            sharpe_max   = ('best_sharpe',  'max'),
            return_max   = ('best_return',  'max'),
        )
        .round(4)
    )
    display(summary)


,seeds,total_evals,cache_end,sharpe_mean,sharpe_max,return_max
round,,,,,,
1,3,5973,5973,1.4600,1.46,29.76
2,3,1461,7434,1.4133,1.47,32.47
